In [17]:
import importlib
import sys

if sys.platform == "emscripten":
    # JupyterLite/Pyodide: install notebook dependencies in the browser kernel.
    get_ipython().run_line_magic("pip", "install -qq pandas faker factory-boy duckdb")
else:
    # Local Python: dependencies must come from uv-managed environment.
    required_modules = ["pandas", "factory", "duckdb"]
    missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
    if missing_modules:
        raise RuntimeError(
            "Missing modules: "
            + ", ".join(missing_modules)
            + ". Run `uv sync` in the project root, then restart the kernel."
        )

In [18]:
import pandas as pd
import factory

In [19]:
class PatientFactory(factory.Factory):
    class Meta:
        model = dict

    patient_id = factory.Sequence(lambda n: f"PT{n:04d}")
    age = factory.Faker('random_int', min=18, max=90)

In [20]:
class StudySiteFactory(factory.Factory):
    class Meta:
        model = dict

    site_id = factory.Sequence(lambda n: f"SITE{n:03d}")
    site_name = factory.Faker('company')
    

In [21]:
class StudyVisitFactory(factory.Factory):
    class Meta:
        model = dict

    visit_id = factory.Sequence(lambda n: f"VISIT{n:05d}")
    # Values are injected from StudyFactory to ensure concrete IDs, not declarations.
    patient_id = None
    site_id = None
    visit_date = factory.Faker('date_this_year')
    specimen_schema = None

In [22]:
import random

SPECIMEN_TYPES = [
    "EDTA plasma",
    "Serum",
    "Heparin plasma",
    "PAXgene RNA",
    "Buffy coat",
]

def generate_specimen_schema():
    picked_types = random.sample(SPECIMEN_TYPES, k=random.randint(2, 4))
    return [
        {"specimen_type": specimen_type, "count": random.randint(1, 6)}
        for specimen_type in picked_types
    ]

class StudyFactory(factory.Factory):
    class Meta:
        model = dict

    Name = factory.Faker('sentence', nb_words=3)
    StartDate = factory.Faker('date_this_decade')
    EndDate = factory.Faker('date_this_decade')
    Sites = factory.LazyFunction(lambda: [StudySiteFactory() for _ in range(3)])
    Participants = factory.LazyFunction(lambda: [PatientFactory() for _ in range(10)])
    Visits = factory.LazyAttribute(
        lambda obj: [
            StudyVisitFactory(
                patient_id=random.choice([p['patient_id'] for p in obj.Participants]),
                site_id=random.choice([s['site_id'] for s in obj.Sites]),
                specimen_schema=generate_specimen_schema(),
            )
            for _ in range(5)
        ]
    )

In [23]:
# Generate a random dataset
study = StudyFactory()
# Convert to DataFrame for display
study

{'Name': 'Teach.',
 'StartDate': datetime.date(2021, 10, 29),
 'EndDate': datetime.date(2024, 8, 30),
 'Sites': [{'site_id': 'SITE000', 'site_name': 'Mcmillan-Smith'},
  {'site_id': 'SITE001', 'site_name': 'Martinez-Whitaker'},
  {'site_id': 'SITE002', 'site_name': 'Donovan, Garcia and Morgan'}],
 'Participants': [{'patient_id': 'PT0000', 'age': 63},
  {'patient_id': 'PT0001', 'age': 67},
  {'patient_id': 'PT0002', 'age': 48},
  {'patient_id': 'PT0003', 'age': 21},
  {'patient_id': 'PT0004', 'age': 82},
  {'patient_id': 'PT0005', 'age': 64},
  {'patient_id': 'PT0006', 'age': 52},
  {'patient_id': 'PT0007', 'age': 41},
  {'patient_id': 'PT0008', 'age': 46},
  {'patient_id': 'PT0009', 'age': 21}],
 'Visits': [{'visit_id': 'VISIT00000',
   'patient_id': 'PT0007',
   'site_id': 'SITE000',
   'visit_date': datetime.date(2026, 3, 26),
   'specimen_schema': [{'specimen_type': 'PAXgene RNA', 'count': 4},
    {'specimen_type': 'Heparin plasma', 'count': 2}]},
  {'visit_id': 'VISIT00001',
   'pa

In [24]:
import sys
from pathlib import Path

def resolve_duckdb_file() -> str:
    # In JupyterLite/Pyodide, use a local file in the virtual FS.
    if sys.platform == "emscripten":
        return "clinical_study.db"

    # In desktop Python, keep using the repo's content/data location.
    db_path = (Path.cwd() / "../data/clinical_study.db").resolve()
    db_path.parent.mkdir(parents=True, exist_ok=True)
    return str(db_path)

def resolve_data_dir() -> Path:
    # In JupyterLite/Pyodide, write under a local data folder in the virtual FS.
    if sys.platform == "emscripten":
        data_path = (Path.cwd() / "data").resolve()
    else:
        data_path = (Path.cwd() / "../data").resolve()

    data_path.mkdir(parents=True, exist_ok=True)
    return data_path

duckdb_file = resolve_duckdb_file()
data_dir = resolve_data_dir()
print(f"DuckDB file: {duckdb_file}")

DuckDB file: C:\Users\dunn0172\Documents\GitHub\biorepository-data-wrangling\content\data\clinical_study.db


In [25]:
# Delete existing database file if it exists
import os
if os.path.exists(duckdb_file):
    os.remove(duckdb_file)

In [26]:
# Write generated study to DuckDB database
import duckdb
# Create a connection to DuckDB
conn = duckdb.connect(duckdb_file)


In [27]:
# Create tables and insert data
conn.execute("""
CREATE TABLE IF NOT EXISTS Study (
    Name TEXT,
    StartDate DATE,
    EndDate DATE
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS StudySite (
    site_id TEXT,
    site_name TEXT
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS Patient (
    patient_id TEXT,
    age INTEGER
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS StudyVisit (
    visit_id TEXT,
    patient_id TEXT,
    site_id TEXT,
    visit_date DATE
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS StudyVisitSpecimen (
    visit_id TEXT,
    specimen_type TEXT,
    specimen_count INTEGER
)""")
conn.execute("""CREATE TABLE IF NOT EXISTS StudySpecimen (
    specimen_id TEXT,
    visit_id TEXT,
    specimen_type TEXT,
    aliquot_number INTEGER
)""")


In [28]:
# Make reruns idempotent by clearing previous synthetic data first.
conn.execute("DELETE FROM StudySpecimen")
conn.execute("DELETE FROM StudyVisitSpecimen")
conn.execute("DELETE FROM StudyVisit")
conn.execute("DELETE FROM Patient")
conn.execute("DELETE FROM StudySite")
conn.execute("DELETE FROM Study")

# Insert study data
conn.execute("INSERT INTO Study (Name, StartDate, EndDate) VALUES (?, ?, ?)", (study['Name'], study['StartDate'], study['EndDate']))
# Insert site data
for site in study['Sites']:
    conn.execute("INSERT INTO StudySite (site_id, site_name) VALUES (?, ?)", (site['site_id'], site['site_name']))
# Insert patient data
for patient in study['Participants']:
    conn.execute("INSERT INTO Patient (patient_id, age) VALUES (?, ?)", (patient['patient_id'], patient['age']))
# Insert visit and specimen data
for visit in study['Visits']:
    conn.execute(
        "INSERT INTO StudyVisit (visit_id, patient_id, site_id, visit_date) VALUES (?, ?, ?, ?)",
        (visit['visit_id'], visit['patient_id'], visit['site_id'], visit['visit_date'])
    )

    for specimen in visit['specimen_schema']:
        conn.execute(
            "INSERT INTO StudyVisitSpecimen (visit_id, specimen_type, specimen_count) VALUES (?, ?, ?)",
            (visit['visit_id'], specimen['specimen_type'], specimen['count'])
        )

        for aliquot_number in range(1, specimen['count'] + 1):
            specimen_prefix = specimen['specimen_type'].upper().replace(' ', '_')
            specimen_id = f"{visit['visit_id']}_{specimen_prefix}_{aliquot_number:02d}"
            conn.execute(
                "INSERT INTO StudySpecimen (specimen_id, visit_id, specimen_type, aliquot_number) VALUES (?, ?, ?, ?)",
                (specimen_id, visit['visit_id'], specimen['specimen_type'], aliquot_number),
            )
# Commit changes and keep the connection open for verification cells.
conn.commit()

In [29]:
# Read data back from DuckDB to verify
tables = conn.execute("SHOW TABLES").fetchdf()
print("Available tables:")
print(tables)

study_df = conn.execute("SELECT * FROM Study").fetchdf()
sites_df = conn.execute("SELECT * FROM StudySite").fetchdf()
patients_df = conn.execute("SELECT * FROM Patient").fetchdf()
visits_df = conn.execute("SELECT * FROM StudyVisit").fetchdf()
specimen_schema_df = conn.execute("SELECT * FROM StudyVisitSpecimen").fetchdf()
specimen_df = conn.execute("SELECT * FROM StudySpecimen ORDER BY specimen_id").fetchdf()

print("Study Data:")
print(study_df)
print("Sites Data:")
print(sites_df)
print("Patients Data:")
print(patients_df)
print("Visits Data:")
print(visits_df)
print("Specimen Schema Data:")
print(specimen_schema_df)
print("Individual Specimen Data (first 20 rows):")
print(specimen_df.head(20))



Available tables:
                 name
0             Patient
1               Study
2           StudySite
3       StudySpecimen
4          StudyVisit
5  StudyVisitSpecimen
Study Data:
     Name  StartDate    EndDate
0  Teach. 2021-10-29 2024-08-30
Sites Data:
   site_id                   site_name
0  SITE000              Mcmillan-Smith
1  SITE001           Martinez-Whitaker
2  SITE002  Donovan, Garcia and Morgan
Patients Data:
  patient_id  age
0     PT0000   63
1     PT0001   67
2     PT0002   48
3     PT0003   21
4     PT0004   82
5     PT0005   64
6     PT0006   52
7     PT0007   41
8     PT0008   46
9     PT0009   21
Visits Data:
     visit_id patient_id  site_id visit_date
0  VISIT00000     PT0007  SITE000 2026-03-26
1  VISIT00001     PT0003  SITE002 2026-03-23
2  VISIT00002     PT0009  SITE000 2026-01-09
3  VISIT00003     PT0004  SITE002 2026-01-21
4  VISIT00004     PT0006  SITE000 2026-04-04
Specimen Schema Data:
      visit_id   specimen_type  specimen_count
0   VISIT00000     

In [30]:
# Export specimen rows with visit/patient/study/site identifiers.
specimen_export_query = """
SELECT
    ss.specimen_id,
    ss.visit_id,
    sv.patient_id,
    sv.site_id,
    CAST(1 AS INTEGER) AS study_id,
    ss.specimen_type,
    ss.aliquot_number
FROM StudySpecimen ss
INNER JOIN StudyVisit sv
    ON ss.visit_id = sv.visit_id
ORDER BY ss.specimen_id
"""
specimen_export_df = conn.execute(specimen_export_query).fetchdf()
specimen_csv_path = data_dir / "specimens.csv"
specimen_export_df.to_csv(specimen_csv_path, index=False)
print(f"Exported specimen rows to {specimen_csv_path}")

Exported specimen rows to C:\Users\dunn0172\Documents\GitHub\biorepository-data-wrangling\content\data\specimens.csv


In [31]:
# Compare schema-level counts to generated individual specimen counts
query = """
SELECT
    svs.visit_id,
    svs.specimen_type,
    svs.specimen_count AS schema_count,
    COUNT(ss.specimen_id) AS generated_count
FROM StudyVisitSpecimen svs
LEFT JOIN StudySpecimen ss
    ON svs.visit_id = ss.visit_id
   AND svs.specimen_type = ss.specimen_type
GROUP BY svs.visit_id, svs.specimen_type, svs.specimen_count
ORDER BY svs.visit_id, svs.specimen_type
"""
joined_df = conn.execute(query).fetchdf()
print(joined_df)

# Close connection when all reads are complete.
conn.close()

      visit_id   specimen_type  schema_count  generated_count
0   VISIT00000  Heparin plasma             2                2
1   VISIT00000     PAXgene RNA             4                4
2   VISIT00001     EDTA plasma             6                6
3   VISIT00001  Heparin plasma             6                6
4   VISIT00001     PAXgene RNA             3                3
5   VISIT00002     EDTA plasma             3                3
6   VISIT00002     PAXgene RNA             3                3
7   VISIT00003     EDTA plasma             6                6
8   VISIT00003  Heparin plasma             1                1
9   VISIT00003     PAXgene RNA             3                3
10  VISIT00003           Serum             5                5
11  VISIT00004     EDTA plasma             2                2
12  VISIT00004  Heparin plasma             3                3
13  VISIT00004     PAXgene RNA             5                5


In [32]:
# Quick QA: pivot one visit to specimen-type columns
conn = duckdb.connect(duckdb_file)

qa_visit_id = conn.execute(
    "SELECT visit_id FROM StudyVisit ORDER BY visit_id LIMIT 1"
).fetchone()[0]

pivot_query = """
SELECT
    ? AS visit_id,
    SUM(CASE WHEN specimen_type = 'EDTA plasma' THEN 1 ELSE 0 END) AS edta_plasma,
    SUM(CASE WHEN specimen_type = 'Serum' THEN 1 ELSE 0 END) AS serum,
    SUM(CASE WHEN specimen_type = 'Heparin plasma' THEN 1 ELSE 0 END) AS heparin_plasma,
    SUM(CASE WHEN specimen_type = 'PAXgene RNA' THEN 1 ELSE 0 END) AS paxgene_rna,
    SUM(CASE WHEN specimen_type = 'Buffy coat' THEN 1 ELSE 0 END) AS buffy_coat
FROM StudySpecimen
WHERE visit_id = ?
"""

qa_pivot_df = conn.execute(pivot_query, (qa_visit_id, qa_visit_id)).fetchdf()
print(qa_pivot_df)

conn.close()

     visit_id  edta_plasma  serum  heparin_plasma  paxgene_rna  buffy_coat
0  VISIT00000          0.0    0.0             2.0          4.0         0.0
